Do not consider this for evaluation, as I had practiced/did it earlier with one dataset and then, tried to change it. This is mess.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import random

In [2]:
class DualEncoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hid_dim, n_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embed_dim)
        
        self.history_rnn = nn.LSTM(embed_dim, hid_dim, n_layers, dropout=dropout)
        self.document_rnn = nn.LSTM(embed_dim, hid_dim, n_layers, dropout=dropout)

        self.dropout = nn.Dropout(dropout)

    def forward(self, history, document):
        embed_hist = self.dropout(self.embedding(history))
        embed_doc = self.dropout(self.embedding(document))

        hist_output, (hidden_h, cell_h) = self.history_rnn(embed_hist)

        doc_output, (hidden_d, cell_d) = self.document_rnn(embed_doc)

        return hist_output, hidden_h, cell_h, doc_output, hidden_d, cell_d
        


In [3]:
class DualCoAttentionDecoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, enc_hid_dim, dec_hid_dim, n_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim)

        self.attention_hist = nn.Linear(enc_hid_dim + dec_hid_dim, dec_hid_dim)
        self.v_hist = nn.Linear(dec_hid_dim, 1, bias=False)

        self.attention_doc = nn.Linear(enc_hid_dim + dec_hid_dim, dec_hid_dim)
        self.v_doc = nn.Linear(dec_hid_dim, 1, bias = False)

        self.rnn = nn.LSTM(emb_dim + 2 * enc_hid_dim, dec_hid_dim, n_layers, dropout)

        self.fc_out = nn.Linear(emb_dim + dec_hid_dim + (enc_hid_dim)*2, vocab_size)

        self.dropout = nn.Dropout(dropout)
    
    def calc_attention(self, hidden, encoder_outputs, attn_layer, v_layer):
        batch_size = encoder_outputs.shape[1]
        src_len = encoder_outputs.shape[0]

        hidden = hidden.squeeze(0).unsqueeze(1).repeat(1, src_len, 1)

        encoder_outputs = encoder_outputs.permute(1, 0, 2)

        en = torch.tanh(attn_layer(torch.cat((hidden, encoder_outputs), dim = 2)))

        attention = v_layer(en).squeeze(2)

        return F.softmax(attention, dim=1)

    def forward(self, input, hidden, cell, hist_out, doc_out):
        input = input.unsqueeze(0)

        embedded = self.dropout(self.embedding(input))

        a_hist = self.calc_attention(hidden[-1:], hist_out, self.attn_hist, self.v_hist)
        c_hist = torch.bmm(a_hist.unsqueeze(1), hist_out.permute(1, 0, 2)).permute(1, 0, 2)
        
        a_doc = self.calc_attention(hidden[-1:], doc_out, self.attn_doc, self.v_doc)
        c_doc = torch.bmm(a_doc.unsqueeze(1), doc_out.permute(1, 0, 2)).permute(1, 0, 2)

        rnn_input = torch.cat((embedded, c_hist, c_doc), dim = 2)
        output, (hidden, cell) = self.rnn(rnn_input, hidden, cell)

        predictions = self.fc_out(torch.cat((output.squeeze(0), c_hist.squeeze(0), c_doc.squeeze(0), embedded.squeeze(0)), dim=1))

        return predictions, hidden, cell


In [4]:

class Attention(nn.Module):
    def __init__(self, enc_hid_dim, dec_hid_dim):
        super().__init__()
        
        self.attn = nn.Linear(enc_hid_dim + dec_hid_dim, dec_hid_dim)
        self.v = nn.Linear(dec_hid_dim, 1, bias=False)

    def forward(self, hidden, enc_outputs):
        batch_size = enc_outputs.shape[1]
        src_len = enc_outputs.shape[0]

        hidden = hidden.squeeze(0).unsqueeze(1).repeat(1, src_len, 1)

        enc_outputs = enc_outputs.permute(1, 0, 2)

        en = torch.tanh(self.attn(torch.cat((hidden, enc_outputs), dim=2)))


        attention = self.v(en).squeeze(2)

        return F.softmax(attention, dim=1)

In [5]:
class DualSeq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device
        
    def forward(self, src, trg, teacher_forcing=0.5):
        batch_size = src.shape[1]
        trg_len = trg.shape[0]
        trg_vocab_size = self.decoder.output_dim
        
        outputs = torch.zeros(trg_len, batch_size, trg_vocab_size).to(self.device)
        
        out_hist, hid_hist, cell_hist, out_doc, hid_doc, cell_doc = self.encoder(src)
        
        hidden = hid_hist + hid_doc
        cell = cell_hist + cell_doc
        input = trg[0, :]
        
        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(input, hidden, cell, out_hist, out_doc)
            outputs[t] = output
            
            teacher_force = random.random() < teacher_forcing
            top1 = output.argmax(1) 
            input = trg[t] if teacher_force else top1
            
        return outputs

In [6]:
!pip install datasets

In [ ]:
from datasets import load_dataset
import string
from torch.utils.data import Dataset, DataLoader
from collections import Counter
from torch.nn.utils.rnn import pad_sequence

print("Loading dataset...")
dataset = load_dataset("festvox/cmu_hinglish_dog")
train_data = dataset['train']
val_data = dataset['validation']
test_data = dataset['test']


Loading dataset...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/844k [00:00<?, ?B/s]